**WHAT IS LANGGRAPH?**

Langgraph is an open source AI agent framework designed to build, deploy and manage complex generative AI agent workflows. It provides a set of tools and libraries that enable users to create, run and optimize large language models in a scalable and efficient manner. Langgraph uses the power of graph-based architectures to model and manage the intricate relationships between various components of an AI agent workflow.

LangGraph allows us to code this relay race.

Nodes: These are the agents or functions.

Edges: These are the rules of who goes next.

State: This is the shared memory.

THE SETUP:

I'm using Ollama here to run a local LLM (Llama 3). This means i doesn't need any OpenAI API key to follow along

In [ ]:
# Ollama + LangGraph Setup for Google Colab

# Step 1: Install Ollama
print("📦 Installing Ollama...")
!curl -fsSL https://ollama.com/install.sh | sh


# Step 2: Start Ollama server in the background
print("🚀 Starting Ollama server...")
import subprocess
import time
import requests

# Start Ollama server as a background process
ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)
# Wait for server to be ready (check if it responds)
print("⏳ Waiting for Ollama server to be ready...")
max_retries = 30
for i in range(max_retries):
    try:
        response = requests.get("http://localhost:11434")
        if response.status_code == 200:
            print("✅ Ollama server is ready!")
            break
    except:
        pass
    time.sleep(1)
    if i % 5 == 0:
        print(f"   Still waiting... ({i+1}s)")
else:
    print("⚠️ Server may not be ready, but continuing anyway...")


# Step 3: Pull the Llama3 model
print("\n⬇️ Pulling llama3.2:1b model (smaller and faster for Colab)...")
!ollama pull llama3.2:1b

# Give it a moment to settle
time.sleep(3)

# Verify the model was downloaded
print("\n📋 Verifying installed models:")
!ollama list

# Step 4: Install Python dependencies
print("\n📚 Installing Python libraries...")
!pip install -q langgraph langchain langchain-community langchain-ollama ddgs

print("\n✨ Setup complete! You can now use Ollama with LangGraph.\n")
print("=" * 60)
print("✅ Ready to build your LangGraph application!")
print("=" * 60)

📦 Installing Ollama...
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading Linux amd64 bundle
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
🚀 Starting Ollama server...
⏳ Waiting for Ollama server to be ready...
✅ Ollama server is ready!

⬇️ Pulling llama3.2:1b model (smaller and faster for Colab)...


📋 Verifying installed models:
NAME             ID              SIZE      MODIFIED       
llama3.2:1b      baf6a787fdff    1.3 GB    3 seconds ago     
llama3:latest    365c0bd3c000    4.7 GB    17 minutes ago    

📚 Installing Python libraries...

✨ Setup complete! You can now use Ollama with LangGraph.

✅ Ready to build your LangGraph application!


Step 1: Defining the Shared State

In [ ]:
from typing import TypedDict, List
from langgraph.graph import StateGraph, END
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.tools.ddg_search import DuckDuckGoSearchRun


In [ ]:
class AgentState(TypedDict):
    topic: str
    research_data: List[str]
    blog_post: str

Step 2: The Researcher Agent

In [ ]:
def researcher_node(state: AgentState):
    topic = state["topic"]
    print(f"🔍 Researcher is looking up: {topic}...")

    search = DuckDuckGoSearchRun()

    try:
        results = search.run(f"key facts and latest news about {topic}")
    except Exception as e:
        results = f"Could not find data: {e}"

    print("✅ Research complete.")
    return {"research_data": state.get("research_data", []) + [results]}

Step 3: The Writer Agent

In [ ]:
# ChatOllama from langchain-ollama package
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate

def writer_node(state: AgentState):
    print("✍️ Writer is drafting the post...")

    topic = state["topic"]
    data = state["research_data"][-1] if state["research_data"] else ""

    # Use smaller model with conservative settings
    llm = ChatOllama(
        model="llama3.2:1b",
        temperature=0.7,
        num_predict=512,  # Limit output length
        timeout=60  # Add timeout
    )

    prompt = ChatPromptTemplate.from_template(
        """You are a tech blog writer.
Write a short, engaging blog post (200-300 words) about "{topic}"
based on the following research data:

{data}

Keep it concise and focused."""
    )

    chain = prompt | llm

    try:
        response = chain.invoke({"topic": topic, "data": data})
        blog_content = response.content
    except Exception as e:
        print(f"⚠️ Error generating blog post: {e}")
        blog_content = f"Error: Unable to generate blog post. {str(e)}"

    print("✅ Writing complete.")
    return {"blog_post": blog_content}

Step 4: Wiring the Graph

In [ ]:
# ----- Build the LangGraph -----
workflow = StateGraph(AgentState)

workflow.add_node("Researcher", researcher_node)
workflow.add_node("Writer", writer_node)

workflow.set_entry_point("Researcher")
workflow.add_edge("Researcher", "Writer")
workflow.add_edge("Writer", END)

app = workflow.compile()

Step 5: Running the System

In [ ]:
if __name__ == "__main__":
    print("\n" + "="*60)
    print("🚀 Starting the Multi-Agent System...")
    print("="*60 + "\n")

    inputs: AgentState = {
        "topic": "The future of AI Agents",
        "research_data": [],
        "blog_post": "",
    }

    try:
        result = app.invoke(inputs)

        print("\n" + "="*60)
        print("📝 FINAL BLOG POST")
        print("="*60 + "\n")
        print(result["blog_post"])
        print("\n" + "="*60)
    except Exception as e:
        print(f"\n❌ Error running the system: {e}")
        print("\nTroubleshooting tips:")
        print("1. Restart the Colab runtime")
        print("2. Try: !ollama run llama3.2:1b 'Hello' to test the model")
        print("3. Check Colab resources (RAM/GPU usage)")


🚀 Starting the Multi-Agent System...

🔍 Researcher is looking up: The future of AI Agents...
✅ Research complete.
✍️ Writer is drafting the post...
✅ Writing complete.

📝 FINAL BLOG POST

**The Rise of Agentic AI: What to Expect in 2025**

As we enter the final stretch of 2024, a pressing question looms large: what does the future hold for AI agents? The latest research suggests that agentic AI is poised to revolutionize industries and transform our lives.

A Salesforce study reveals a staggering 282% increase in AI adoption by CIOs, with many organizations experimenting with AI agents as workers. This trend is not only driving business outcomes but also improving employee experience, according to a survey of 62% respondents.

But what can we realistically expect from agentic AI in 2025? While some implementations may falter, research points to several promising areas:

* **Improved decision-making**: Agentic AI agents are becoming increasingly sophisticated, enabling more accurate an